* Primeiramente vamos fazer uma inspeção para saber mais detalhes dos datasets que vamos usar no projeto.

In [1]:
import glob
import os

import pandas as pd

arquivos_csv = glob.glob("/Users/Lased/Documents/RotaCerta/data/raw/*.csv")

def checagem_dfs(arquivos_csv):
    info_arquivos = {}

    for arquivo in arquivos_csv:
        df = pd.read_csv(arquivo)
        linhas, colunas = df.shape
        info_arquivos[arquivo] = (linhas, colunas)

    return info_arquivos

infos = checagem_dfs(arquivos_csv)
for arquivo, (linhas, colunas) in infos.items():
    nome_arquivo = os.path.basename(arquivo)
    print(f"Arquivo: {nome_arquivo} - Linhas: {linhas}, colunas: {colunas}")

Arquivo: avaliacoes.csv - Linhas: 25000, colunas: 4
Arquivo: clientes.csv - Linhas: 8000, colunas: 5
Arquivo: entregas.csv - Linhas: 25000, colunas: 6
Arquivo: pedidos.csv - Linhas: 25000, colunas: 10


* Vamos fazer uma inspeção para entender mais sobre os dados que vamos trabalhar nesse projeto.

In [ ]:
import numpy as np


def inspecao(df, nome_dataset="Dataset"):

    """Realiza um profiling exploratório inicial de um DataFrame.

    Imprime um relatório com shape, tipos de dados, valores nulos,
    checagem de colunas de data, duplicatas, estatísticas descritivas
    das colunas numéricas, cardinalidade das colunas categóricas/booleanas
    e a janela temporal das colunas de data.

    Args:
        df (pd.DataFrame): DataFrame a ser inspecionado.
        nome_dataset (str, opcional): Nome de exibição do dataset no
            cabeçalho do relatório. Padrão: "Dataset".

    Returns:
        None: a função apenas imprime o relatório no console/notebook,
        não retorna nenhum valor.
    """
    
    print("-=" * 30)
    print(f"INSPEÇÃO: {nome_dataset.upper()}")
    
    # 1. Shape:
    print(f"\n Shape do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas")

    # 2. Tipos de Dados:
    print("\n Tipo de dado de cada coluna:")
    print(df.dtypes)

    # 3. Valores nulos(quantidade e porcentagem):
    nulos = df.isnull().sum()
    if nulos.sum() > 0:
        df_nulos = pd.DataFrame({'Qtd_Nulos': nulos, 'Porcentagem': (nulos / len(df)) * 100})
        print("\n Valores nulos encontrados no dataset:")
        print(df_nulos)
    else:
        print("\n Nenhum valor nulo encontrado.")

    # 4. Checando as colunas de data para saber se são ou não objetos datetime:
    print("\n Checagem das colunas de data:")
    # Procura qualquer coluna que tenha a palavra "data" no nome
    colunas_de_data = [col for col in df.columns if 'data' in col.lower()]

    if colunas_de_data:
        for col in colunas_de_data:
            # Checa o tipo e já imprime o resultado
            is_datetime = pd.api.types.is_datetime64_any_dtype(df[col]) 
            print(f"-> A coluna {col} está no formato datetime? {is_datetime}") # True = a coluna é datetime | False = a coluna não é datetime
    else:
        print("-> Nenhuma coluna com a palavra 'data' foi encontrada.")

    # 5. Checagem de duplicatas:
    print(f"\n Linhas totalmente duplicadas: {df.duplicated().sum()}")

    # 6. Ranges e Sanidade Numérica:
    colunas_numericas = df.select_dtypes(include=np.number).columns
    if len(colunas_numericas) > 0:
        print("\n Estatísticas Descritivas (Númericas):")
        print(df[colunas_numericas].describe().round(2))

    # 7. Cardinalidade de Categóricas:
    colunas_categoricas = df.select_dtypes(include=['object', 'category', 'bool']).columns
    if len(colunas_categoricas) > 0:
        print("\n Cardinalidade(Categóricas/Booleanas):")
        for col in colunas_categoricas:
            valores_unicos = df[col].nunique()
            print(f" -> {col}: {valores_unicos} categorias exclusivas")
            # Mostrar o top 3 de valores:
            top3 = df[col].value_counts(normalize=True).head(3) * 100
            print(f" Top categorias: {top3.round(2).to_dict()}")

    # 8. Limites de Tempo:
    if colunas_de_data:
        print("\n Janela de tempo(Datas):")
        for col in colunas_de_data:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                print(f" -> {col}: de {df[col].min()} até {df[col].max()}")
            else:
                print(f"-> A coluna '{col}' precisa ser convertida para datetime!")
    

    print("-=" * 30)